# 04 · Iterative calculation (circular references)

A levered model where **interest depends on debt** and **debt is repaid
with cash flow after interest** — a textbook circular reference. We
resolve it with `enable_iterative_calculation=True`.

## The circular structure

- `interest(t) = debt(t) * rate`

- `debt(t) = debt(t-1) - cash_sweep(t)`

- `cash_sweep(t) = ebitda(t) - interest(t)`  ← depends on interest



All three are ordinary `@row` methods. When the solver hits the cycle
mid-iteration it substitutes the previous pass's value (seeded at `0.0`
on the first pass), so no special seeding is required.

In [ ]:
from dataclasses import dataclass

from finmodel import Model, row, PredefinedFormats as F, PredefinedStyles as S



@dataclass

class Inputs:

    ebitda: float

    opening_debt: float

    rate: float



class LeveredModel(Model[Inputs]):

    @row(group="P&L", format=F.USD)

    def ebitda(self, t):

        return self.inputs.ebitda



    # Circular row: interest <- debt <- cash sweep <- interest

    @row(group="P&L", format=F.USD)

    def interest(self, t):

        return self.debt(t) * self.inputs.rate



    @row(group="Cash", format=F.USD)

    def cash_sweep(self, t):

        # Cash available to repay debt, after paying interest.

        return max(self.ebitda(t) - self.interest(t), 0)



    @row(group="Balance", format=F.USD)

    def debt(self, t):

        if t == 0:

            return self.inputs.opening_debt

        return max(self.debt(t - 1) - self.cash_sweep(t), 0)

## Without the solver it raises

By default a self-referential period raises a clear `ValueError`.

In [ ]:
inputs = Inputs(ebitda=300, opening_debt=1_000, rate=0.08)

try:

    m = LeveredModel(periods=6, inputs=inputs)

    m.calculate()

except ValueError as e:

    print("Raised as expected:\n", e)

## Enable iterative calculation

Turn on the solver and give it convergence settings. The engine iterates
all rows until the largest cell-to-cell change drops below `threshold`.
Note `interest` in period 0 correctly resolves to `debt(0) * rate`.

In [ ]:
model = LeveredModel(

    periods=6,

    inputs=inputs,

    enable_iterative_calculation=True,

    threshold=1e-8,

    max_iterations=200,

    damping=0.8,

)

model.calculate()

model.show()

## Inspect the converged series

In [ ]:
print("debt:    ", model.get_result_data("debt").round(2))

print("interest:", model.get_result_data("interest").round(2))

print("sweep:   ", model.get_result_data("cash_sweep").round(2))